# Maze complexity measures

Generates the maze set (original lateralized / non-lateralized mazes plus 100 new hemifield-balanced ones), creates start/goal variants for each layout, and computes the measures used later in the analysis:

1. layout measures (symmetry, obstacle spacing, obstacle shape, density / entropy)
2. optimal path measures
3. value-guided construal (VGC)
4. a structure-level summary (one row per base layout)

Shared helpers (`to_grid`, `get_obstacle_labels`, `shortest_path`) live in `maze_utils.py`.

**Inputs** (in `mazes/`): `mazes_Lateralized.json`, `mazes_Nonlateralized.json`

**Outputs:** `mazes_generated.json` and `mazes_sg_variants.json` in `mazes/`; `maze_complexity_metrics.csv`, `maze_with_path_complexity_metrics.csv` and `structure_metrics.csv` in `metrics/`

Maze encoding: `.` free, `#` wall, `0`-`5` obstacles, `S` start, `G` goal.

## 0. Setup

In [ ]:
import os
import json
import random
import multiprocessing
from itertools import combinations
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

from vgc_project.vgc import value_guided_construal

from maze_utils import DIRS4, to_grid, get_obstacle_labels, shortest_path

In [ ]:
# file names used throughout
MAZE_DIR = "mazes"        # maze layouts (JSON)
METRICS_DIR = "metrics"   # complexity measures (CSV)
os.makedirs(MAZE_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

LATERALIZED_JSON = os.path.join(MAZE_DIR, "mazes_Lateralized.json")
NONLATERALIZED_JSON = os.path.join(MAZE_DIR, "mazes_Nonlateralized.json")
GENERATED_JSON = os.path.join(MAZE_DIR, "mazes_generated.json")
VARIANTS_JSON = os.path.join(MAZE_DIR, "mazes_sg_variants.json")
METRICS_CSV = os.path.join(METRICS_DIR, "maze_complexity_metrics.csv")
PATH_METRICS_CSV = os.path.join(METRICS_DIR, "maze_with_path_complexity_metrics.csv")
STRUCTURE_CSV = os.path.join(METRICS_DIR, "structure_metrics.csv")


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1)

## 1. Maze generation

All new mazes share the same template: the fixed cross-shaped wall in the middle, S in the bottom-left and G in the top-right. Six obstacles (labelled `0`-`5`) are then dropped in, each one a contiguous blob of `blocks_per_obstacle` cells that doesn't overlap anything else.

In [ ]:
TEMPLATE = [
    "..........G",
    "...........",
    "...........",
    ".....#.....",
    ".....#.....",
    "...#####...",
    ".....#.....",
    ".....#.....",
    "...........",
    "...........",
    "S..........",
]

### 1.1 Shared helpers

In [ ]:
def _parse_template(template: List[str]) -> Tuple[int, int, Set[Tuple[int, int]], Tuple[int, int], Tuple[int, int]]:
    H = len(template)
    W = len(template[0])
    walls = set()
    start = goal = None

    for r, row in enumerate(template):
        if len(row) != W:
            raise ValueError("All rows in template must have the same length.")
        for c, ch in enumerate(row):
            if ch == "#":
                walls.add((r, c))
            elif ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
            elif ch != ".":
                # template should only have walls + S/G, anything else is a mistake
                raise ValueError(f"Template contains non-wall non-empty char '{ch}' at {(r, c)}. "
                                 "Use '.' '#' 'S' 'G' only in template.")

    if start is None or goal is None:
        raise ValueError("Template must contain exactly one S and one G.")
    return H, W, walls, start, goal


def _neighbors4(r: int, c: int, H: int, W: int):
    if r > 0:
        yield (r - 1, c)
    if r < H - 1:
        yield (r + 1, c)
    if c > 0:
        yield (r, c - 1)
    if c < W - 1:
        yield (r, c + 1)


def _grow_contiguous_shape(
    seed: Tuple[int, int],
    blocks: int,
    H: int,
    W: int,
    forbidden: Set[Tuple[int, int]],
    rng: random.Random,
) -> Set[Tuple[int, int]]:
    """
    Grow a connected set of `blocks` cells from `seed`, staying inside the grid
    and off any forbidden cell. Returns an empty set if it gets stuck.
    """
    shape = {seed}
    frontier = {seed}

    while len(shape) < blocks:
        candidates = set()
        for (r, c) in frontier:
            for nb in _neighbors4(r, c, H, W):
                if nb not in shape and nb not in forbidden:
                    candidates.add(nb)

        if not candidates:
            return set()

        new_cell = rng.choice(tuple(candidates))
        shape.add(new_cell)
        frontier.add(new_cell)

        # stop the frontier from blowing up
        if len(frontier) > 50:
            frontier = set(rng.sample(list(frontier), 25))

    return shape


def _render_grid(H, W, walls, start, goal, placements):
    grid = [["." for _ in range(W)] for _ in range(H)]
    for (r, c) in walls:
        grid[r][c] = "#"
    grid[start[0]][start[1]] = "S"
    grid[goal[0]][goal[1]] = "G"
    for lab, cells in placements.items():
        for (r, c) in cells:
            grid[r][c] = lab
    return ["".join(row) for row in grid]

### 1.2 Hemifield-balanced generator

Puts 3 obstacles on each side (left/right) and 3 in each half (top/bottom). The split by quadrant is UL = `a`, UR = `3-a`, LL = `3-a`, LR = `a`, and every obstacle has to sit fully inside its quadrant.

In [ ]:
def generate_maze_hemifield(
    template,
    blocks_per_obstacle,
    obstacle_labels=None,
    seed=None,
    a=1,
    max_tries=5000,
):
    if obstacle_labels is None:
        obstacle_labels = list("012345")
    if len(obstacle_labels) != 6:
        raise ValueError("Need exactly 6 obstacle labels.")
    if a < 0 or a > 3:
        raise ValueError("a must be in {0,1,2,3}. Recommended: 1 or 2.")

    rng = random.Random(seed)
    H, W, walls, start, goal = _parse_template(template)
    mid_r, mid_c = H // 2, W // 2

    base_forbidden = set(walls) | {start, goal}

    def quadrant_cells(rows, cols):
        return [(r, c) for r in rows for c in cols if (r, c) not in base_forbidden]

    quad_cells = {
        "UL": quadrant_cells(range(0, mid_r), range(0, mid_c)),
        "UR": quadrant_cells(range(0, mid_r), range(mid_c, W)),
        "LL": quadrant_cells(range(mid_r, H), range(0, mid_c)),
        "LR": quadrant_cells(range(mid_r, H), range(mid_c, W)),
    }
    target = {"UL": a, "UR": 3 - a, "LL": 3 - a, "LR": a}

    # shuffle labels, then hand them out to quadrants in order
    labels = obstacle_labels[:]
    rng.shuffle(labels)
    quad_labels = {}
    idx = 0
    for q in ["UL", "UR", "LL", "LR"]:
        quad_labels[q] = labels[idx: idx + target[q]]
        idx += target[q]

    def in_quadrant(shape, q):
        if not shape:
            return False
        for (r, c) in shape:
            if q == "UL" and not (r < mid_r and c < mid_c): return False
            if q == "UR" and not (r < mid_r and c >= mid_c): return False
            if q == "LL" and not (r >= mid_r and c < mid_c): return False
            if q == "LR" and not (r >= mid_r and c >= mid_c): return False
        return True

    def place_in_quadrant(q, candidate_cells, occupied):
        for _ in range(500):
            if not candidate_cells:
                return None
            seed_cell = rng.choice(candidate_cells)
            shape = _grow_contiguous_shape(seed_cell, blocks_per_obstacle, H, W, occupied, rng)
            if shape and len(shape) == blocks_per_obstacle and in_quadrant(shape, q):
                return shape
        return None

    for _ in range(max_tries):
        placements = {}
        occupied = set(base_forbidden)

        ok = True
        for q in ["UL", "UR", "LL", "LR"]:
            for lab in quad_labels[q]:
                free = [c for c in quad_cells[q] if c not in occupied]
                shape = place_in_quadrant(q, free, occupied)
                if shape is None:
                    ok = False
                    break
                placements[lab] = shape
                occupied |= shape
            if not ok:
                break

        if ok:
            return _render_grid(H, W, walls, start, goal, placements)

    raise RuntimeError("Failed to generate maze under both hemifield constraints. "
                       "Try smaller blocks_per_obstacle or set a=1 or a=2.")

In [ ]:
def generate_many(
    n: int,
    template: List[str],
    blocks_per_obstacle: int,
    obstacle_labels: List[str] = None,
    seed: int = None,
    start_index: int = 12,
) -> Dict[str, List[str]]:
    """
    Generate n mazes keyed grid-XXX-l. start_index=12 so the new ids carry on
    after the 12 original lateralized mazes (grid-000-l ... grid-011-l).
    """
    rng = random.Random(seed)
    out = {}
    for i in range(start_index, start_index + n):
        out[f"grid-{i:03d}-l"] = generate_maze_hemifield(
            template=template,
            blocks_per_obstacle=blocks_per_obstacle,
            obstacle_labels=obstacle_labels,
            seed=rng.randint(0, 10**9),
        )
    return out

In [ ]:
# quick check
example = generate_maze_hemifield(TEMPLATE, blocks_per_obstacle=4, seed=2)
print("\n".join(example))

## 2. Build the maze set

Original 12 lateralized + 12 non-lateralized mazes, plus 100 new hemifield-balanced ones (`grid-012-l` to `grid-111-l`).

In [ ]:
with open(LATERALIZED_JSON, "r", encoding="utf-8") as f:
    mazes = json.load(f)

with open(NONLATERALIZED_JSON, "r", encoding="utf-8") as f:
    nonlateralized_mazes = json.load(f)

generated_mazes = generate_many(100, TEMPLATE, blocks_per_obstacle=4, seed=123)

mazes.update(nonlateralized_mazes)
mazes.update(generated_mazes)
print(len(mazes), "base mazes")

# keep a copy of the new mazes - the same seed gives different mazes on other Python versions
save_json(generated_mazes, GENERATED_JSON)

## 3. Start/goal variants

For every base layout, keep the walls and obstacles and resample S and G: 100 variants per maze, at least 10 apart (Manhattan) and with G reachable from S. Variants are named `<base_id>-SGxx`.

In [ ]:
def generate_start_goal_variants(
    maze,
    n_variants=5,
    min_manhattan=10,
    require_reachable=True,
    use_shortest_path_distance=False,
    seed=None,
    max_tries=20000,
):
    """
    Same walls/obstacles, new S and G.

    min_manhattan              - minimum Manhattan distance between S and G
    require_reachable          - only keep variants where G is reachable from S
    use_shortest_path_distance - apply min_manhattan to the shortest-path length
                                 instead (stricter, slower)
    """
    rng = random.Random(seed)
    grid = np.array([list(row) for row in maze], dtype="<U1")
    H, W = grid.shape

    # clear the old S/G, then only place on free floor
    grid[grid == "S"] = "."
    grid[grid == "G"] = "."
    empties = [(r, c) for r in range(H) for c in range(W) if grid[r, c] == "."]
    if len(empties) < 2:
        return []

    variants = []
    tries = 0
    while len(variants) < n_variants and tries < max_tries:
        tries += 1
        s = rng.choice(empties)
        g = rng.choice(empties)
        if s == g:
            continue
        if abs(s[0] - g[0]) + abs(s[1] - g[1]) < min_manhattan:
            continue

        cand = grid.copy()
        cand[s] = "S"
        cand[g] = "G"
        cand_maze = ["".join(cand[r, :]) for r in range(H)]

        if require_reachable or use_shortest_path_distance:
            path = shortest_path(cand_maze)
            if path is None:
                continue
            if use_shortest_path_distance and len(path) - 1 < min_manhattan:
                continue

        variants.append(cand_maze)

    return variants


def generate_variants_for_all(
    mazes_dict,
    n_variants_per_maze=5,
    min_manhattan=10,
    require_reachable=True,
    use_shortest_path_distance=False,
    seed=123,
):
    rng = random.Random(seed)
    out = {}
    for maze_id, maze in mazes_dict.items():
        variants = generate_start_goal_variants(
            maze,
            n_variants=n_variants_per_maze,
            min_manhattan=min_manhattan,
            require_reachable=require_reachable,
            use_shortest_path_distance=use_shortest_path_distance,
            seed=rng.randint(0, 10**9),
        )
        for i, v in enumerate(variants):
            out[f"{maze_id}-SG{i:02d}"] = v
    return out

In [ ]:
mazes_variants = generate_variants_for_all(
    mazes,
    n_variants_per_maze=100,
    min_manhattan=10,
    require_reachable=True,
)
mazes.update(mazes_variants)
save_json(mazes_variants, VARIANTS_JSON)

maze_complexity_metrics = pd.DataFrame({
    "maze_id": list(mazes.keys()),
    "mazes": list(mazes.values()),
})
# base layout each variant came from (base mazes map to themselves)
maze_complexity_metrics["base_id"] = (
    maze_complexity_metrics["maze_id"].astype(str).str.split("-SG").str[0]
)
print(len(maze_complexity_metrics), "mazes in total (base + variants)")

## 4. Layout measures

These only depend on the walls and obstacles (S and G are ignored), so they're computed once per base layout (124 mazes) and then copied to all of that layout's S/G variants.

### 4.1 Symmetry

In [ ]:
def is_obstacle(ch):
    return ch.isdigit()


def compute_horizontal_symmetry(maze):
    """Proportion of cells in the left half that match their mirror cell on the right (obstacle vs not)."""
    H, W = len(maze), len(maze[0])
    half = W // 2
    matches = sum(
        is_obstacle(maze[r][c]) == is_obstacle(maze[r][W - 1 - c])
        for r in range(H) for c in range(half)
    )
    return matches / (H * half)


def compute_vertical_symmetry(maze):
    """Same idea, mirrored top/bottom."""
    H, W = len(maze), len(maze[0])
    half = H // 2
    matches = sum(
        is_obstacle(maze[r][c]) == is_obstacle(maze[H - 1 - r][c])
        for r in range(half) for c in range(W)
    )
    return matches / (half * W)

### 4.2 Obstacle spacing

Distance between two obstacles = smallest Manhattan distance between any pair of their cells.

In [ ]:
def compute_dist(maze, obstacle_1, obstacle_2):
    grid = to_grid(maze)
    A = np.argwhere(grid == obstacle_1)
    B = np.argwhere(grid == obstacle_2)
    if len(A) == 0 or len(B) == 0:
        return np.inf
    return np.abs(A[:, None, :] - B[None, :, :]).sum(axis=-1).min()


def compute_mean_distance(maze):
    obstacles = get_obstacle_labels(maze)
    if len(obstacles) < 2:
        return 0.0
    dists = [compute_dist(maze, a, b) for a, b in combinations(obstacles, 2)]
    return sum(dists) / len(dists)


def distance_variance(maze, ddof=1):
    pairs = list(combinations(get_obstacle_labels(maze), 2))
    if len(pairs) < 2:
        return 0.0
    dists = [compute_dist(maze, a, b) for a, b in pairs]
    return float(np.var(dists, ddof=ddof))


def _nn_distances(maze):
    """distance from each obstacle to its closest neighbour"""
    obstacles = get_obstacle_labels(maze)
    return [min(compute_dist(maze, a, b) for b in obstacles if b != a) for a in obstacles]


def nearest_neighbor_distance_mean(maze):
    if len(get_obstacle_labels(maze)) < 2:
        return 0.0
    return float(np.mean(_nn_distances(maze)))


def nearest_neighbor_distance_variance(maze, ddof=1):
    if len(get_obstacle_labels(maze)) < 3:
        return 0.0
    return float(np.var(_nn_distances(maze), ddof=ddof))

### 4.3 Obstacle shape

In [ ]:
def obstacle_similarity(maze, obstacle_a, obstacle_b):
    """
    Translation-invariant shape similarity between two obstacles, in [0, 1]:
    the best Jaccard overlap (shared cells / cells in either) over all relative
    shifts of the two shapes. 1 means identical shapes.
    """
    grid = to_grid(maze)
    A = np.argwhere(grid == obstacle_a)
    B = np.argwhere(grid == obstacle_b)
    if A.size == 0 or B.size == 0:
        return 0.0

    # move both shapes to their own top-left corner
    A0 = A - A.min(axis=0)
    B0 = B - B.min(axis=0)
    setA = set(map(tuple, A0))
    setB0 = set(map(tuple, B0))

    Ay, Ax = A0.max(axis=0)
    By, Bx = B0.max(axis=0)
    dy_range = range(-int(By), int(Ay) + 1)
    dx_range = range(-int(Bx), int(Ax) + 1)

    best_shape_sim = 0.0
    for dy in dy_range:
        for dx in dx_range:
            shiftedB = {(y + dy, x + dx) for (y, x) in setB0}
            inter = len(setA & shiftedB)
            if inter == 0:
                continue
            j = inter / len(setA | shiftedB)
            if j > best_shape_sim:
                best_shape_sim = j
                if best_shape_sim == 1.0:
                    break
        if best_shape_sim == 1.0:
            break

    return float(best_shape_sim)


def compute_mean_sim(maze):
    obstacles = get_obstacle_labels(maze)
    if len(obstacles) < 2:
        return 0.0
    sims = [obstacle_similarity(maze, a, b) for a, b in combinations(obstacles, 2)]
    return sum(sims) / len(sims)


def count_unique_obstacle(maze):
    """Number of distinct obstacle shapes (translation only, no rotation/reflection)."""
    grid = to_grid(maze)
    shapes = []
    for label in get_obstacle_labels(maze):
        coords = np.argwhere(grid == label)
        if coords.size == 0:
            continue
        coords0 = coords - coords.min(axis=0)
        shapes.append(tuple(sorted(map(tuple, coords0))))
    return len(set(shapes))

In [ ]:
def obstacle_roughness(grid, label):
    """perimeter / area (4-neighbour perimeter)"""
    coords = np.argwhere(grid == label)
    if coords.size == 0:
        return 0.0

    cells = set(map(tuple, coords))
    perimeter = sum(
        (y + dy, x + dx) not in cells
        for y, x in cells for dy, dx in DIRS4
    )
    return perimeter / len(cells)


def compute_mean_roughness(maze):
    grid = to_grid(maze)
    obstacles = get_obstacle_labels(maze)
    if not obstacles:
        return 0.0
    values = [obstacle_roughness(grid, ob) for ob in obstacles]
    return sum(values) / len(values)

### 4.4 Density and local entropy

Both work on a binary obstacle mask. Walls count as free space; the wall is the same cross in every maze.

- **edge density**: share of neighbouring cell pairs that cross an obstacle/free boundary
- **local entropy**: binary entropy of the obstacle proportion in each k x k window, averaged over all windows (bits, 0-1)

In [ ]:
def _obstacle_mask(maze):
    return np.vectorize(str.isdigit)(to_grid(maze))


def _boundary_density(mask):
    mask = mask.astype(int)
    H, W = mask.shape
    diff_h = (mask[:, 1:] != mask[:, :-1]).sum()
    diff_v = (mask[1:, :] != mask[:-1, :]).sum()
    return (diff_h + diff_v) / (H * (W - 1) + (H - 1) * W)


def _mean_window_entropy(mask, k):
    mask = mask.astype(np.float32)
    H, W = mask.shape
    if k > H or k > W:
        return 0.0

    p = np.lib.stride_tricks.sliding_window_view(mask, (k, k)).mean(axis=(-1, -2))

    # entropy is 0 for p = 0 or 1, only compute in between
    ent = np.zeros_like(p)
    mid = (p > 0) & (p < 1)
    ent[mid] = -(p[mid] * np.log2(p[mid]) + (1 - p[mid]) * np.log2(1 - p[mid]))
    return float(ent.mean())


def edge_density(maze):
    return _boundary_density(_obstacle_mask(maze))


def mean_local_entropy(maze, window=3):
    return _mean_window_entropy(_obstacle_mask(maze), window)

### 4.5 Compute everything

Computed on the base layouts only, then merged onto every variant by `base_id`.

In [ ]:
layout_metrics = {
    "horizontal_symmetry": compute_horizontal_symmetry,
    "vertical_symmetry": compute_vertical_symmetry,
    "n_unique_obs": count_unique_obstacle,
    "mean_sim": compute_mean_sim,
    "mean_dis": compute_mean_distance,
    "edge_density": edge_density,
    "local_entropy_3": mean_local_entropy,
    "mean_roughness": compute_mean_roughness,
    "var_pairwise_dist": distance_variance,
    "var_nn_dist": nearest_neighbor_distance_variance,
    "mean_nn_distance": nearest_neighbor_distance_mean,
}

base_layouts = maze_complexity_metrics.loc[
    maze_complexity_metrics["maze_id"] == maze_complexity_metrics["base_id"],
    ["base_id", "mazes"],
]

layout_df = base_layouts[["base_id"]].copy()
for col, fn in tqdm(layout_metrics.items(), desc="Layout measures"):
    layout_df[col] = base_layouts["mazes"].apply(fn)

maze_complexity_metrics = maze_complexity_metrics.merge(layout_df, on="base_id", how="left")
layout_df

## 5. Optimal path measures

`shortest_path` (in `maze_utils.py`) runs A* over (cell, previous direction), so among equally short paths the one with the fewest turns is picked. From that path: its length, the number of turns, and turns per step.

In [ ]:
def path_metrics_from_path(path):
    if path is None or len(path) < 2:
        return {
            "optimal_path_length": np.nan,
            "n_turns": np.nan,
            "turn_to_path_ratio": np.nan,
        }

    path_length = len(path) - 1
    steps = [(path[i + 1][0] - path[i][0], path[i + 1][1] - path[i][1]) for i in range(path_length)]
    n_turns = sum(steps[i] != steps[i - 1] for i in range(1, len(steps)))

    return {
        "optimal_path_length": float(path_length),
        "n_turns": float(n_turns),
        "turn_to_path_ratio": float(n_turns / path_length),
    }

In [ ]:
maze_complexity_metrics["optimal_path"] = maze_complexity_metrics["mazes"].apply(shortest_path)

path_metrics = (
    maze_complexity_metrics["optimal_path"]
    .apply(path_metrics_from_path)
    .apply(pd.Series)
)
maze_complexity_metrics = pd.concat([maze_complexity_metrics, path_metrics], axis=1)

maze_complexity_metrics.to_csv(METRICS_CSV, index=False)
maze_complexity_metrics

## 6. Value-guided construal

Run VGC on every solvable maze and keep the construal with the highest value of representation. This is by far the slowest step, so the results are written to `metrics/maze_with_path_complexity_metrics.csv` as soon as it finishes.

In [ ]:
INV_TEMP = 10
PLANNING_ALG = "policy_iteration"


def compute_vgc(tile_array):
    res = value_guided_construal(
        tile_array=tile_array,
        construal_inverse_temp=INV_TEMP,
        planning_alg=PLANNING_ALG,
    )
    vor = res["value_of_representation"]
    opt_c = max(vor, key=vor.get)
    return opt_c, len(opt_c), vor

In [ ]:
# only mazes where G is reachable
maze_with_path_complexity_metrics = maze_complexity_metrics[
    maze_complexity_metrics["optimal_path_length"].notna()
].copy()

maze_list = maze_with_path_complexity_metrics["mazes"].tolist()

# leave a few cores free so the machine stays usable
n_jobs = max(multiprocessing.cpu_count() - 3, 1)

results = Parallel(n_jobs=n_jobs)(
    delayed(compute_vgc)(m) for m in tqdm(maze_list, desc="Computing VGC")
)
optimal_construals, optimal_sizes, vors = zip(*results)

maze_with_path_complexity_metrics["optimal_construal"] = optimal_construals
maze_with_path_complexity_metrics["optimal_construal_size"] = optimal_sizes
maze_with_path_complexity_metrics["vgc_value_of_representation"] = vors

maze_with_path_complexity_metrics.to_csv(PATH_METRICS_CSV, index=False)

## 7. Structure-level summary

One row per base layout. Layout measures are the same across a layout's variants so the first value is taken; path and construal measures are averaged over the variants.

In [ ]:
structure_metrics = (
    maze_with_path_complexity_metrics
    .groupby("base_id")
    .agg(
        # layout measures (identical across variants)
        horizontal_symmetry=("horizontal_symmetry", "first"),
        vertical_symmetry=("vertical_symmetry", "first"),
        n_unique_obs=("n_unique_obs", "first"),
        edge_density=("edge_density", "first"),
        local_entropy_3=("local_entropy_3", "first"),
        mean_sim=("mean_sim", "first"),
        mean_dis=("mean_dis", "first"),
        mean_roughness=("mean_roughness", "first"),
        mean_nn_distance=("mean_nn_distance", "first"),
        var_pairwise_dist=("var_pairwise_dist", "first"),
        var_nn_dist=("var_nn_dist", "first"),

        # averaged over S/G variants
        mean_optimal_path_length=("optimal_path_length", "mean"),
        mean_n_turns=("n_turns", "mean"),
        mean_turn_to_path_ratio=("turn_to_path_ratio", "mean"),
        mean_optimal_construal_size=("optimal_construal_size", "mean"),

        n_variants=("maze_id", "count"),
    )
    .reset_index()
)

structure_metrics.to_csv(STRUCTURE_CSV, index=False)
structure_metrics.head()